In [1]:
import pandas as pd, numpy as np, joblib, time, warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
print("✅ Imports done")

✅ Imports done


In [2]:
bundle = joblib.load("models/fraud_xgb.joblib")
model, TH = bundle["model"], bundle["threshold"]

df = pd.read_csv("data/transactions_clean.csv", parse_dates=["ts"])
df = df.sort_values("ts").reset_index(drop=True)

def add_features(df):
    df = df.copy()
    df["log_amount"]           = np.log1p(df["amount"])
    df["avg_tx_amt_24h"]       = df["prev_24h_amt_card"] / (df["prev_24h_tx_count_card"] + 1e-3)
    df["velocity_ratio"]       = df["velocity_amt_1h"] / (df["avg_tx_amt_24h"] + 1e-3)
    df["amount_vs_24h_zscore"] = (df["amount"] - df["avg_tx_amt_24h"]) / (df["avg_tx_amt_24h"] + 1e-3)
    df["is_weekend"]           = df["dayofweek"].isin([5, 6]).astype(int)
    df["high_velocity_flag"]   = (df["prev_1h_tx_count_card"] > 3).astype(int)
    df["merchant_cat_rare"]    = 0
    return df

CUT   = int(len(df) * 0.80)
ref   = add_features(df.iloc[:CUT].copy())   # train = reference window
curr  = add_features(df.iloc[CUT:].copy())   # valid = current window

NUM  = ["amount","log_amount","prev_24h_tx_count_card","prev_24h_amt_card",
        "prev_1h_tx_count_card","velocity_amt_1h","avg_tx_amt_24h","velocity_ratio",
        "amount_vs_24h_zscore","hour","dayofweek",
        "is_international","is_night","is_weekend","merchant_cat_rare","high_velocity_flag"]
CAT  = ["merchant_cat","city","country","device_type","channel"]

X_ref  = ref[NUM+CAT];  y_ref  = ref["is_fraud"]
X_curr = curr[NUM+CAT]; y_curr = curr["is_fraud"]
print("✅ Data loaded")

✅ Data loaded


In [3]:
proba   = model.predict_proba(X_curr)[:,1]
pr_auc  = average_precision_score(y_curr, proba)
roc_auc = roc_auc_score(y_curr, proba)
fpr, tpr, _ = roc_curve(y_curr, proba)
recall_1fpr = tpr[np.searchsorted(fpr, 0.01)]

print("─" * 45)
print("  MODEL PERFORMANCE")
print("─" * 45)
print(f"  PR-AUC          : {pr_auc:.4f}  {'✅' if pr_auc > 0.30 else '⚠️ Low'}")
print(f"  ROC-AUC         : {roc_auc:.4f}")
print(f"  Recall @FPR=1%  : {recall_1fpr:.4f}")

─────────────────────────────────────────────
  MODEL PERFORMANCE
─────────────────────────────────────────────
  PR-AUC          : 0.9670  ✅
  ROC-AUC         : 0.9957
  Recall @FPR=1%  : 0.9455


In [4]:
def compute_psi(ref_data, curr_data, n_bins=10):
    bins = np.linspace(
        min(ref_data.min(), curr_data.min()),
        max(ref_data.max(), curr_data.max()),
        n_bins + 1
    )
    bins[0], bins[-1] = -np.inf, np.inf
    ref_pct  = (np.histogram(ref_data,  bins=bins)[0] + 1e-6) / len(ref_data)
    curr_pct = (np.histogram(curr_data, bins=bins)[0] + 1e-6) / len(curr_data)
    return round(float(np.sum((curr_pct - ref_pct) * np.log(curr_pct / ref_pct))), 4)

monitor_features = ["log_amount","velocity_ratio","avg_tx_amt_24h","prev_1h_tx_count_card","hour"]

print(f"{'Feature':<30} {'PSI':>8}  Status")
print("─" * 50)
for feat in monitor_features:
    psi = compute_psi(X_ref[feat].values, X_curr[feat].values)
    status = "🔴 RETRAIN" if psi > 0.20 else ("🟡 MONITOR" if psi > 0.10 else "🟢 Stable")
    print(f"{feat:<30} {psi:>8.4f}  {status}")

Feature                             PSI  Status
──────────────────────────────────────────────────
log_amount                       0.0022  🟢 Stable
velocity_ratio                   0.0070  🟢 Stable
avg_tx_amt_24h                   0.0036  🟢 Stable
prev_1h_tx_count_card            0.0011  🟢 Stable
hour                             0.0030  🟢 Stable


In [5]:
latencies = []
for _ in range(30):
    sample = X_curr.sample(1)
    t0 = time.time()
    model.predict_proba(sample)
    latencies.append((time.time() - t0) * 1000)

p50, p95, p99 = np.percentile(latencies, [50, 95, 99])
SLO = 150  # ms

print(f"  p50 latency : {p50:.1f} ms")
print(f"  p95 latency : {p95:.1f} ms  {'✅ SLO OK' if p95 < SLO else '⚠️ SLO BREACH'}")
print(f"  p99 latency : {p99:.1f} ms")
print(f"  Target SLO  : p95 < {SLO} ms")

  p50 latency : 10.1 ms
  p95 latency : 11.8 ms  ✅ SLO OK
  p99 latency : 12.2 ms
  Target SLO  : p95 < 150 ms


In [6]:
cols = df.columns.tolist()
pii_keywords = ["pan","card_number","cvv","ssn","email","phone","name","dob","aadhaar"]
pii_found    = [c for c in cols if any(p in c.lower() for p in pii_keywords)]
hashed       = [c for c in cols if "hash" in c.lower()]

print("PII Governance Audit")
print(f"  Raw PII columns   : {pii_found if pii_found else '✅ None found'}")
print(f"  Hashed IDs        : {hashed}")
print(f"  Model features    : {len(X_curr.columns)}")
print(f"  PII in model      : {'❌ Remove!' if pii_found else '✅ Clean'}")
print()
print("Retraining Policy:")
print("  • Weekly rolling 8-week window")
print("  • Trigger: PSI > 0.20 OR PR-AUC drops > 10%")
print("  • Must backtest 2 weeks before promotion")

PII Governance Audit
  Raw PII columns   : ✅ None found
  Hashed IDs        : ['merchant_id_hash', 'card_id_hash']
  Model features    : 21
  PII in model      : ✅ Clean

Retraining Policy:
  • Weekly rolling 8-week window
  • Trigger: PSI > 0.20 OR PR-AUC drops > 10%
  • Must backtest 2 weeks before promotion
